In [ ]:
import json
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript

# =========================================================
# GREEN + WHITE + BLACK UI
# =========================================================
display(HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700;800&display=swap');

:root{
    --bg:#000000;
    --panel:#0d0d0d;
    --panel2:#151515;
    --line:#2a2a2a;
    --text:#f5f5f5;
    --muted:#a5a5a5;
    --accent:#00c853;
    --accent-soft:#39e27d;
    --radius:14px;
}

body, .jp-Notebook, .jp-Cell, .jp-OutputArea-output{
    background:var(--bg)!important;
    color:var(--text)!important;
    font-family:'Inter',sans-serif!important;
}

/* inputs */
.widget-text input,
.widget-textarea textarea{
    background:var(--panel)!important;
    color:var(--text)!important;
    border:1px solid var(--line)!important;
    border-radius:12px!important;
    padding:10px 12px!important;
    font-size:13px!important;
    box-shadow:none!important;
}

.widget-text input:focus,
.widget-textarea textarea:focus{
    border-color:var(--accent)!important;
    box-shadow:0 0 0 2px rgba(0,200,83,.14)!important;
}

/* buttons */
.widget-button button{
    border:none!important;
    border-radius:10px!important;
    font-size:13px!important;
    font-weight:700!important;
    padding:9px 16px!important;
    box-shadow:none!important;
    transition:all .18s ease!important;
}

.widget-button button:hover{
    filter:brightness(1.06)!important;
    transform:translateY(-1px)!important;
}

.widget-button button:active{
    transform:translateY(0)!important;
}

/* generate button */
.widget-button button.mod-primary,
.widget-button button.jupyter-widgets.jupyter-button.widget-button.mod-primary{
    background:linear-gradient(135deg,var(--accent),var(--accent-soft))!important;
    color:#000000!important;
}

/* tabs */
.widget-tab .p-TabBar-tab{
    background:var(--panel2)!important;
    color:var(--muted)!important;
    border:1px solid var(--line)!important;
    border-bottom:none!important;
    border-radius:10px 10px 0 0!important;
    padding:10px 18px!important;
    font-size:13px!important;
    font-weight:700!important;
}

.widget-tab .p-TabBar-tab:hover{
    color:var(--text)!important;
}

.widget-tab .p-TabBar-tab.p-mod-current{
    background:var(--panel)!important;
    color:var(--text)!important;
    box-shadow:inset 0 2px 0 var(--accent)!important;
}

.widget-tab > .p-Widget{
    background:var(--panel)!important;
    border:1px solid var(--line)!important;
    border-radius:0 0 14px 14px!important;
    padding:14px!important;
}

/* labels */
.ui-title{
    font-size:26px;
    font-weight:800;
    color:var(--text);
    margin:0 0 14px 0;
    letter-spacing:.2px;
}

.ui-title span{
    color:var(--accent);
}

.ui-label{
    font-size:11px;
    color:var(--muted);
    margin:14px 0 6px 2px;
    text-transform:uppercase;
    letter-spacing:1.2px;
    font-weight:700;
}

.small-label{
    font-size:11px;
    color:var(--muted);
    margin:0 0 6px 2px;
    text-transform:uppercase;
    letter-spacing:1.2px;
    font-weight:700;
}

.status-text{
    font-size:12px;
    color:var(--accent-soft);
    margin-top:8px;
}

/* prompt text areas */
.prompt-area textarea{
    font-family:Consolas, monospace!important;
    line-height:1.4!important;
}
</style>
"""))

# =========================================================
# HELPERS
# =========================================================
def safe_value(v, fallback):
    v = (v or "").strip()
    return v if v else fallback

def copy_to_clipboard(text):
    js_safe = json.dumps(text)
    display(Javascript(f"navigator.clipboard.writeText({js_safe});"))

def flash(btn):
    old_desc = btn.description
    old_style = btn.button_style
    btn.description = "Copied"
    btn.button_style = "success"
    import threading
    def reset():
        btn.description = old_desc
        btn.button_style = old_style
    threading.Timer(1.2, reset).start()

def clear_outputs():
    prompt1_area.value = ""
    prompt2_area.value = ""
    prompt3_area.value = ""
    prompt4_area.value = ""
    prompt5_area.value = ""

# =========================================================
# INPUTS
# =========================================================
TEST1FORPROMPT1 = widgets.Textarea(
    placeholder='Paste reference script...',
    layout=widgets.Layout(width='100%', height='240px')
)

TEST1FORPROMPT2 = widgets.Text(
    placeholder='Write video title...',
    layout=widgets.Layout(width='100%')
)

TEST1FORPROMPT3 = widgets.Text(
    value='3000',
    placeholder='3000',
    layout=widgets.Layout(width='180px')
)

TEST2FORPROMPT3 = widgets.Text(
    value='English',
    placeholder='English',
    layout=widgets.Layout(width='180px')
)

generate_btn = widgets.Button(
    description="Generate",
    button_style='primary',
    layout=widgets.Layout(width='160px', height='42px')
)

status_bar = widgets.HTML("<div class='status-text'>Ready</div>")

# =========================================================
# PROMPTS
# =========================================================
PROMPT_1 = """RESPONSE STYLE
- Short paragraphs + bullet points only.
- No long essays.
- Each subsection: max 6–8 bullets.
- Concrete, prescriptive.

TASK
Analyze ONLY Reference Script 1 (ignore timestamps) and extract its style/structure/rhythm DNA.
Output a reusable “Custom Style & Structure Guide” enabling ≥80–90% perceived similarity.
Do NOT: write a new story, plot-summarize for its own sake, ask questions.

REFERENCE SCRIPT 1
[REFERENCE_SCRIPT_START]
[TEST1FORPROMPT1]
[REFERENCE_SCRIPT_END]

STEP 1 — GLOBAL SNAPSHOT (max 6–8 bullets)
- Niche/sub-niche
- Implied audience (age, knowledge, emotional need)
- Purpose blend (warn/shock/comfort/instruct/inspire/entertain)
- Mood band

STEP 2 — WORD COUNT & MACRO STRUCTURE
2.1 Baseline word count
- If [REFERENCE_WORD_COUNT] is a number: set X=[REFERENCE_WORD_COUNT].
- If [REFERENCE_WORD_COUNT]="auto": estimate cleaned narration word count; set X=estimate.
- Output EXACT line:
  Reference Script 1 Estimated Total Word Count: ~[X words]
- Future scripts: Hook fixed ≈100 words (90–110). Remaining (X−100) across 4 acts.

2.2 Hook + 4 Acts
For Hook + Act I–IV, output (each):
- Function (1 line)
- Approx % of total
- Approx words
- Dominant emotion

STEP 3 — VOICE & ENGAGEMENT
3.1 Voice & Emotion (max 6 bullets)
3.2 Narrator Persona (max 6 bullets)
3.3 Engagement Mechanics (max 6–8 bullets)
End Step 3 with a 3–5 sentence “Voice Profile” paragraph (how they sound; how it feels to listen).

STEP 4 — MICRO-STYLE (sentences/paragraphs/evidence)
4.1 Sentences & Rhythm (max 6–8 bullets)
4.2 Paragraphs (max 4–6 bullets)
4.3 Word Choice / Imagery / Evidence (max 6–10 bullets)

STEP 5 — CUSTOM STYLE & STRUCTURE GUIDE (FINAL)
Headings + bullets only (except 1 short Voice Profile paragraph).
Include sections:

VOICE PROFILE
- 1 short paragraph
- 3–6 bullets (who speaks, closeness, read-aloud sound)

STRUCTURAL BLUEPRINT (Hook + 4 Acts)
- Hook ≈100 words (90–110) fixed; remaining words split across Act I–IV.
- For each part: role, typical content, approx share/words, tension growth.

SENTENCE & PARAGRAPH BLUEPRINT
- 6–10 abstract sentence patterns
- Rules for average length, short punches, one-line paragraphs

WORD & EVIDENCE RULES
- Register, sensory detail, metaphors allowed
- What breaks illusion (slang/memes/meta-YouTube)
- Frequency of numbers/dates/names/quotes + typical attribution phrases

PACING & SIGNATURE ELEMENTS
- How to alternate scenes vs data; calm vs shock; questions vs answers
- Where peaks sit (usually Act II–III)
- 5–10 signature traits

REPLICATION TEST
- “Drop-in Test” description
- State: similarity should feel ≥80–90%

FINAL LINES (MANDATORY)
Reference Script 1 Estimated Total Word Count: ~[X words]
Use this as the default total word-count target for future scripts in this style (unless the user explicitly overrides it).



"""

PROMPT_2 = """RESPONSE STYLE
Short paragraphs + bullets. Compact, practical.

LANGUAGE
Use [TARGET_LANGUAGE]. If [TARGET_LANGUAGE]="auto", use the user’s/main language only. Do not switch.

ROLE
You are the SAME model that produced the Prompt 1 Style Guide. Use it as the ONLY style authority.
Do NOT re-analyze the reference or invent a new style.

INPUTS
New video title: [TEST1FORPROMPT2]
Target total word count: [TEST1FORPROMPT3]

SET TOTAL_USER (HARD)
- If [TARGET_WORD_COUNT]="auto": TOTAL_USER = X from Prompt 1 final line.
- If numeric: TOTAL_USER = [TARGET_WORD_COUNT].

LOCKED WORDCOUNT TARGETS (HARD)
- HOOK_TARGET = 100 (EXACT)
- BODY_USER = max(TOTAL_USER - HOOK_TARGET, 0)

COMPUTE [SECTIONS_COUNT] (body only; do NOT ask user)
Goal: fewer sections, still reliable in Claude Sonnet 4.5.
- TARGET_SECTION_SIZE (words):
  - If TOTAL_USER <= 2000: 900
  - Else if TOTAL_USER <= 4500: 1100
  - Else if TOTAL_USER <= 8000: 1350
  - Else if TOTAL_USER <= 12000: 1500
  - Else if TOTAL_USER <= 18000: 1700
  - Else: 1900
- [SECTIONS_COUNT] = ceil(BODY_USER / TARGET_SECTION_SIZE)
- Clamp: minimum 1, maximum 10

ACT TARGETS (structure only)
- Act I target ≈ round(BODY_USER / 4)
- Act II target ≈ round(BODY_USER / 4)
- Act III target ≈ round(BODY_USER / 4)
- Act IV target ≈ BODY_USER - (Act I + Act II + Act III)

SECTION TARGETS (EXACT, HARD)
Compute exact per-section word targets so the full script lands EXACTLY on TOTAL_USER:
- SECTION_BASE = floor(BODY_USER / [SECTIONS_COUNT])
- REMAINDER = BODY_USER - (SECTION_BASE * [SECTIONS_COUNT])

For each section k = 1..[SECTIONS_COUNT]:
- If k <= REMAINDER:
  Sk_TARGET = SECTION_BASE + 1
- Else:
  Sk_TARGET = SECTION_BASE

Hard rules for Prompt 3:
- Hook variants: exactly 100 words each.
- Each section k: exactly Sk_TARGET words (EXACT).
- No ranges. No “approx”.

TOP SUMMARY (output these lines first)
Target script total requested by user: [TOTAL_USER words] (EXACT)
Hook target: 100 words (EXACT)
Body target (without hooks): [BODY_USER words] (EXACT)
Number of sections (body only): [SECTIONS_COUNT]
Act I target: ~[A1 words]
Act II target: ~[A2 words]
Act III target: ~[A3 words]
Act IV target: ~[A4 words]

STEP 1 — ACT-LEVEL OVERVIEW
For Hook + Act I–IV: 4–6 bullets each (function, emotion, typical start/end, what must be established).

STEP 2 — KEY BEATS BY PART
For Hook + Act I–IV: 5–10 bullets each (mystery/contradiction, escalation, reveals, evidence style, closing aftertaste).

SECTION PLAN FOR PROMPT 3 (MUST include exact targets)
For each Section k=1..[SECTIONS_COUNT], output:
- Exact target: [Sk_TARGET] words (EXACT)
- Act focus (e.g., late Act I + early Act II)
- MUST cover beats (2–5 bullets)
- MUST NOT do yet (no full resolution except final section)

STEP 3 — SHORT STYLE CHECKLIST (for Prompt 3)
- Voice
- Rhythm
- Word choice bans
- Evidence density + attribution phrases
- Pacing/peaks
- 5–8 signature traits that MUST appear


"""

PROMPT_3 = """OUTPUT FORMAT
- Output ONLY narrative script text + the fixed English labels below.
- No headings, bullets, or “Act/Chapter” labels.
- Start immediately with story text (no meta/openers like “Here is…”).

FIXED ENGLISH LINES
Hook variant 1
Hook variant 2
Hook variant 3
Section [BLOCK_ID] of [SECTIONS_COUNT]
Story Complete

Input: [BLOCK_ID] = BLOCK_HOOK]

LANGUAGE
- Input: [TARGET_LANGUAGE] = [TEST2FORPROMPT3]
- If missing/empty/undefined/"auto": set [TARGET_LANGUAGE]="English".
- If explicitly set: write ALL story text (hooks/sections) in exactly [TARGET_LANGUAGE].
- Never ask the user about language.

ROLE
You are the SAME model that made: Prompt 1 Style Guide + Prompt 2 Outline/Section Plan.
Use them as the ONLY authority. Do NOT re-analyze reference. Do NOT ask for any inputs.
Assume TOTAL_USER, BODY_USER, SECTIONS_COUNT, and each section’s Sk_TARGET exist from Prompt 2.

DEFINITIONS (HARD)
- Command "HOOKS" means: generate 3 hook variants ONLY (no sections).
- Command "block k" / "section k" / "BLOCK_ID=k" / "k" means: generate Section k ONLY.
- Command "block a-b" / "section a-b" / "BLOCK_ID=a-b" / "a-b" means: generate Sections a..b ONLY (range mode).
- Hook variants (1/2/3) are NOT sections and MUST NEVER be treated as section numbers.

STATELESS NUMBERING (HARD)
- Do NOT track progress or auto-advance section numbers.
- The ONLY source of truth is the last user command message.

CONTROL INPUT PARSING (HARD)
Treat the LAST user message as the control command.
IMPORTANT: If it contains extra text, ignore everything except a valid control pattern below.

1) RANGE MODE (highest priority)
- If the last message contains a valid range command:
  "block a-b" OR "section a-b" OR "BLOCK_ID=a-b" OR just "a-b"
  where a and b are integers, a<=b, and both in 1..[SECTIONS_COUNT],
  then set START=a and END=b → RANGE MODE.

2) SECTION MODE
- Else, if the last message contains a valid section command:
  "block k" OR "section k" OR "BLOCK_ID=k"
  where k is an integer in 1..[SECTIONS_COUNT],
  then set K=k → SECTION MODE.
- Else, if the entire last message is exactly an integer k in 1..[SECTIONS_COUNT],
  then set K=k → SECTION MODE.

3) HOOKS MODE
- Else, if the last message contains the standalone token "HOOKS" (case-insensitive),
  set MODE=HOOKS.

4) DEFAULT (NO QUESTIONS)
- If none match, default to HOOKS MODE.

ABSOLUTE SINGLE-MODE RULE (HARD)
In ONE answer:
- HOOKS MODE: output exactly 3 hooks and STOP.
- SECTION MODE: output exactly 1 section and STOP.
- RANGE MODE: output sections START..END only (no hooks) and STOP.
No explanations, no questions, no onboarding, no system talk.

BANNED OUTPUT (HARD)
You MUST NEVER output:
- Any questions
- Any instructions like “Please provide BLOCK_ID…”
- Any meta commentary (no “I’m ready…”, no “I will…”)
If uncertain, follow DEFAULT and generate HOOKS.

HOOKS MODE (EXACT)
Generate 3 alternative hook variants for the SAME underlying story.
WORD COUNT (HARD, EXACT):
- Each hook variant MUST be exactly 100 words.
OUTPUT ORDER (exactly this order):
(hook narrative)
Hook variant 1
(hook narrative)
Hook variant 2
(hook narrative)
Hook variant 3
Then STOP immediately. Do NOT generate sections.

SECTION / RANGE MODES (HARD GUARDRAILS)
- No hooks in Section/Range modes.
- Do NOT output: "Hook variant 1/2/3".
- Do NOT repeat the hook. Section 1 starts AFTER the hook.

WORD COUNT ENFORCEMENT (HARD, INTERNAL)
Word counting rule: split on spaces; numbers count as one word; hyphenated counts as one.
Do NOT include the final label line in the word count.

For any section k:
- Read Sk_TARGET(k) from Prompt 2 output for that section.
- Draft to ~92–95% of Sk_TARGET(k) first (do NOT exceed Sk_TARGET yet).
- Add short micro-detail sentences to land EXACTLY on Sk_TARGET(k).
- If you exceed Sk_TARGET(k), delete full sentences ONLY from the end until EXACT.

FINAL SECTION TOTAL-LOCK (HARD)
If generating the FINAL section (k = [SECTIONS_COUNT]):
- Internally count narrative words in ALL previously generated sections 1..k-1 in this chat.
- Set FINAL_TARGET = BODY_USER - WORDS_PREVIOUS_SECTIONS.
- Use FINAL_TARGET (EXACT) instead of Sk_TARGET(k) if there is any drift.
- Enforce EXACT on the final section.

SECTION MODE OUTPUT
- Generate exactly ONE section: Section K.
- After the narrative text, output on a new line:
Section K of [SECTIONS_COUNT]
- If K = [SECTIONS_COUNT], then add on the next line:
Story Complete
Then STOP.

RANGE MODE OUTPUT
- Generate sections START..END in order.
- Hard cap for stability: generate at most 2 sections per answer if Sk_TARGET(START) >= 1400, otherwise at most 3.
  If the requested range exceeds the cap, silently reduce END to fit the cap.
- After each section narrative, output its label line:
Section k of [SECTIONS_COUNT]
- If the last generated k equals [SECTIONS_COUNT], add:
Story Complete
Then STOP.

CUT-OFF CONTINUATION (HARD)
If the previous answer was cut off mid-section (no "Section k of N" label yet) and the user writes "continue":
- Continue the SAME current section immediately (no repeats, no hooks).
- Finish it to the EXACT target, then output the label line and STOP.

"""

PROMPT_4 = """RESPONSE STYLE
- Diagnostic: short paragraphs +TEST1FORPROMPT4
- Rewrite: ONLY narrative script text + fixed English labels/questions (no bullets/headings/meta).
- Post-analysis: short paragraphs + bullets.

LANGUAGE CONTROL
Use the SAME dominant language as the current draft produced by Prompt 3 (detect once; never switch).
ONLY allowed English lines:
- “Should I rewrite the story to increase structural similarity to the reference? (Yes/No)”
- “Continue with Section k of N? (Yes/No)”
- “Would you like me to analyze the rewritten story, explain what I changed, and estimate the new structural similarity to the reference? (Yes/No)”
- Hook variant 1/2/3
- Section [BLOCK_ID] of [SECTIONS_COUNT]
- Story Complete

ROLE
You are the SAME model that made Prompt 1–3. Use the existing guide/outline/section plan; do NOT re-invent style.

CURRENT DRAFT (internal)
Find the most recent Prompt 3 outputs in this chat:
- 3 hooks labeled Hook variant 1/2/3
- Sections labeled “Section k of N” (ordered by k)
Reconstruct full draft = hooks + all available sections.
If none found: output a short message (story language) saying Prompt 3 must be run first, then STOP.

MODE SELECTION
- If user has NOT answered the rewrite question → DIAGNOSTIC MODE.
- If your last message ended with the rewrite question and user said “Yes” → REWRITE MODE.
- If your last message ended with the post-analysis question and user said “Yes” → POST-ANALYSIS MODE.
If output was cut off and user says “continue/продолжай/keep going”: stay in the SAME mode and continue from where you stopped (no restarts/repeats).

DIAGNOSTIC MODE
Internal: compare draft vs Prompt 1 Style Guide + Prompt 2 Outline/Section Plan; estimate similarity ~[X]%.
Output (story language) with headings + bullets:
- STYLE SIMILARITY DIAGNOSTIC: ~[X]%
- Main strengths (2–4)
- Main structural mismatches/risks (3–6)
- Phase notes: Opening/Hooks; Middle; Ending (bullets)
End with EXACT English line:
Should I rewrite the story to increase structural similarity to the reference? (Yes/No)
Do NOT rewrite in this mode.

REWRITE MODE (stepwise)
Goal: ≥85–90% structural style similarity, without changing core story events.
Length constraints (internal):
- OLD_TOTAL ≈ current draft words
- New total must be within round(OLD_TOTAL*0.97) .. round(OLD_TOTAL*1.03)
Per-section constraints:
- For each original Section k word count OLD_k:
  rewritten section must be within round(OLD_k*0.95) .. round(OLD_k*1.05)
Hooks should remain 90–110 words.

What you MAY change: phrasing/rhythm, minor beat order within a section, evidence placement, small clarity/pacing details.
You MUST NOT: change core events/characters, genre, or factual consistency; copy reference sentences.

Stage 1 (first rewrite-mode message): rewrite all 3 hooks (story language), each followed by its English label line.
Then ask EXACT English line:
Continue with Section 1 of N? (Yes/No)

Stage 2 (for each Section k when user answers Yes):
Output ONLY rewritten Section k narrative, then:
Section k of N
If k < N, ask:
Continue with Section [k+1] of N? (Yes/No)
If user answers No: acknowledge in story language and STOP rewriting.

Stage 3 (final Section N):
After rewritten narrative:
Section N of N
Story Complete
Then ask EXACT English line:
Would you like me to analyze the rewritten story, explain what I changed, and estimate the new structural similarity to the reference? (Yes/No)

POST-ANALYSIS MODE
Internal: compare original vs rewritten vs guide/outline; estimate ~[Y]%.
Output (story language) with headings + bullets:
- STYLE ANALYSIS OF REWRITTEN STORY: ~[Y]%
- Main changes applied (3–6)
- Improvements vs original (3–6)
- Remaining minor differences (optional 2–4)
No further questions unless user asks.

"""

PROMPT_5 = """ROLE
You are a professional YouTube SEO expert specializing in high-CTR titles, optimized descriptions, and keyword-rich tags for educational storytelling videos.

INPUT CONTROL
TARGET_LANGUAGE: [TEST1FORPROMPT5]
- If empty or "auto" → use the language of the script
- If specified → ALL outputs MUST be in this language

TASK
Use the MOST RECENT completed script in this chat (including the selected hook and all sections) as your input.

Generate:

1. 3 HIGH-CTR YouTube titles
2. 1 optimized YouTube description
3. YouTube tags (15–20 tags)

RULES

TITLES
- MUST be between 60–80 characters (STRICT)
- Must be emotionally engaging and curiosity-driven
- Use power words (secret, hidden, discovered, truth, shocking, unexplained, etc.)
- Keep titles natural (NOT spammy or clickbait overload)
- Do NOT copy or slightly modify the original title — fully improve it

DESCRIPTION
- Start with a strong hook (1–2 lines max)
- Clearly explain what the video is about
- Include relevant keywords naturally (SEO optimized)
- Keep it clean and readable (no keyword stuffing)

TAGS
- MUST be 15–20 tags
- TOTAL length MUST be close to 500 characters (450–500 ideal)
- Must be relevant to the topic
- Mix of short and long keywords
- Do NOT repeat keywords
- Do NOT use hashtags
- FORMAT MUST BE EXACTLY:
  tag1,tag2,tag3,tag4,tag5

IMPORTANT
- Do NOT ask for the script
- Do NOT summarize the script
- Do NOT add explanations
- Output ONLY the final result

OUTPUT FORMAT

TITLES:
1.
2.
3.

DESCRIPTION:
[Your description here]

TAGS:
tag1,tag2,tag3,tag4,tag5
"""

# =========================================================
# BUILD PROMPTS
# =========================================================
def build_prompts():
    words = safe_value(TEST1FORPROMPT3.value, "3000")
    lang  = safe_value(TEST2FORPROMPT3.value, "English")
    title = safe_value(TEST1FORPROMPT2.value, "")
    ref   = TEST1FORPROMPT1.value

    p1 = PROMPT_1.replace("[TEST1FORPROMPT1]", ref)

    p2 = (PROMPT_2
          .replace("[TEST1FORPROMPT1002]", lang)
          .replace("[TEST1FORPROMPT2]", title)
          .replace("[TEST1FORPROMPT3]", words))

    p3 = (PROMPT_3
          .replace("[TEST1FORPROMPT3]", words)
          .replace("[TEST2FORPROMPT3]", lang))

    p4 = PROMPT_4.replace("[TEST1FORPROMPT3]", words)

    p5 = PROMPT_5.replace("[TEST1FORPROMPT5]", lang)

    return {
        "Prompt 1": p1,
        "Prompt 2": p2,
        "Prompt 3": p3,
        "Prompt 4": p4,
        "Prompt 5": p5
    }

# =========================================================
# OUTPUT AREAS
# =========================================================
PROMPT_HEIGHT = "390px"

prompt1_area = widgets.Textarea(layout=widgets.Layout(width='100%', height=PROMPT_HEIGHT), value="")
prompt2_area = widgets.Textarea(layout=widgets.Layout(width='100%', height=PROMPT_HEIGHT), value="")
prompt3_area = widgets.Textarea(layout=widgets.Layout(width='100%', height=PROMPT_HEIGHT), value="")
prompt4_area = widgets.Textarea(layout=widgets.Layout(width='100%', height=PROMPT_HEIGHT), value="")
prompt5_area = widgets.Textarea(layout=widgets.Layout(width='100%', height=PROMPT_HEIGHT), value="")

prompt1_area.add_class("prompt-area")
prompt2_area.add_class("prompt-area")
prompt3_area.add_class("prompt-area")
prompt4_area.add_class("prompt-area")
prompt5_area.add_class("prompt-area")

copy_btn_1 = widgets.Button(description="Copy Prompt", button_style='', layout=widgets.Layout(width='130px'))
copy_btn_2 = widgets.Button(description="Copy Prompt", button_style='', layout=widgets.Layout(width='130px'))
copy_btn_3 = widgets.Button(description="Copy Prompt", button_style='', layout=widgets.Layout(width='130px'))
copy_btn_4 = widgets.Button(description="Copy Prompt", button_style='', layout=widgets.Layout(width='130px'))
copy_btn_5 = widgets.Button(description="Copy Prompt", button_style='', layout=widgets.Layout(width='130px'))

def make_tab(area, btn):
    return widgets.VBox(
        [
            widgets.HBox([btn], layout=widgets.Layout(justify_content='flex-end')),
            area
        ],
        layout=widgets.Layout(width='100%', height='450px')
    )

tabs = widgets.Tab(
    children=[
        make_tab(prompt1_area, copy_btn_1),
        make_tab(prompt2_area, copy_btn_2),
        make_tab(prompt3_area, copy_btn_3),
        make_tab(prompt4_area, copy_btn_4),
        make_tab(prompt5_area, copy_btn_5),
    ],
    layout=widgets.Layout(width='100%')
)

for i, name in enumerate(["Prompt 1", "Prompt 2", "Prompt 3", "Prompt 4", "Prompt 5"]):
    tabs.set_title(i, name)

# =========================================================
# ACTIONS
# =========================================================
def update_outputs():
    prompts = build_prompts()
    prompt1_area.value = prompts["Prompt 1"]
    prompt2_area.value = prompts["Prompt 2"]
    prompt3_area.value = prompts["Prompt 3"]
    prompt4_area.value = prompts["Prompt 4"]
    prompt5_area.value = prompts["Prompt 5"]

def on_generate(_):
    update_outputs()
    status_bar.value = "<div class='status-text'>Generated</div>"

def copy_prompt(area, btn, name):
    if not area.value.strip():
        status_bar.value = f"<div class='status-text'>{name} is empty</div>"
        return
    copy_to_clipboard(area.value)
    flash(btn)
    status_bar.value = f"<div class='status-text'>{name} copied</div>"

generate_btn.on_click(on_generate)
copy_btn_1.on_click(lambda b: copy_prompt(prompt1_area, copy_btn_1, "Prompt 1"))
copy_btn_2.on_click(lambda b: copy_prompt(prompt2_area, copy_btn_2, "Prompt 2"))
copy_btn_3.on_click(lambda b: copy_prompt(prompt3_area, copy_btn_3, "Prompt 3"))
copy_btn_4.on_click(lambda b: copy_prompt(prompt4_area, copy_btn_4, "Prompt 4"))
copy_btn_5.on_click(lambda b: copy_prompt(prompt5_area, copy_btn_5, "Prompt 5"))

# =========================================================
# SETTING BOX
# =========================================================
def setting_box(label, widget):
    return widgets.VBox(
        [
            widgets.HTML(f"<div class='small-label'>{label}</div>"),
            widget
        ],
        layout=widgets.Layout(width='190px')
    )

# =========================================================
# LAYOUT
# =========================================================
display(HTML("<div class='ui-title'>Prompt <span>Builder</span></div>"))

display(widgets.HTML("<div class='ui-label'>Reference Script</div>"))
display(TEST1FORPROMPT1)

display(widgets.HTML("<div class='ui-label'>Video Title</div>"))
display(TEST1FORPROMPT2)

display(widgets.HTML("<div class='ui-label'>Settings</div>"))
display(widgets.HBox(
    [
        setting_box("Target Words", TEST1FORPROMPT3),
        setting_box("Language", TEST2FORPROMPT3),
    ],
    layout=widgets.Layout(gap='14px', flex_flow='row wrap')
))

display(widgets.HTML("<div style='height:12px'></div>"))
display(generate_btn)
display(status_bar)

display(widgets.HTML("<div style='height:18px'></div>"))
display(tabs)

# =========================================================
# INIT
# =========================================================
clear_outputs()
status_bar.value = "<div class='status-text'>Ready</div>"

HTML(value="<div class='ui-label'>Reference Script</div>")

Textarea(value='', layout=Layout(height='240px', width='100%'), placeholder='Paste reference script...')

HTML(value="<div class='ui-label'>Video Title</div>")

Text(value='', layout=Layout(width='100%'), placeholder='Write video title...')

HTML(value="<div class='ui-label'>Settings</div>")

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>